# BIG DATA PROJECT - SUJET 1 : L'ARCHITECTE DU STOCKAGE

## SECTION 1 : PROJECT INTRODUCTION

**Objective of the project:**
This project explores the fundamental differences between row-based (CSV) and columnar (Parquet) storage formats in a Big Data ecosystem. The objective is to understand how storage choices impact data processing speeds, storage costs, and overall system efficiency using Apache Spark.

**Difference between CSV and Parquet:**
* **CSV (Comma-Separated Values):** A row-based, human-readable text format. It is easy to use but lacks structure, schema enforcement, and compression efficiency. When querying, the engine must read the entire row even if only one column is needed.
* **Parquet:** A columnar, binary storage format designed for Hadoop ecosystems. It stores data column by column, enabling highly efficient data compression and query optimization.

**Why columnar storage is important in Big Data:**
In Big Data analytics, queries often involve scanning specific columns (e.g., aggregations like average or sum) across massive datasets rather than fetching entire records. Columnar storage allows the query engine to read only the necessary columns, significantly reducing the amount of data processed.

**Expected benefits:**
* **Reduced storage size:** Columnar data allows for better compression algorithms since all data in a column is of the same type.
* **Faster analytics:** Reading only required columns speeds up analytical queries.
* **Reduced disk I/O:** Less data read from disk means fewer I/O bottlenecks.
* **Better bandwidth usage:** Smaller file sizes and selective reading reduce network traffic when transferring data across a cluster.

## SECTION 2 : START SPARK SESSION

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC Taxi Storage Optimization") \
    .getOrCreate()


## SECTION 3 : LOAD DATASET FROM HDFS

In [2]:
# التأكد من أن جلسة Spark ترى HDFSk
try:
    path = "hdfs://namenode:9000/taxi/yellow_tripdata_2021-01.csv"
    # قراءة الملف مع تحديد الخيارات بدقة
    df_csv = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path)
    
    # عرض النتائج
    df_csv.show(5)
    print("تم تحميل البيانات بنجاح!")
except Exception as e:
    print(f"حدث خطأ أثناء القراءة: {e}")
    # طباعة معلومات الاتصال للتصحيح
    print(f"Master URL: {spark.sparkContext.master}")

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2021-01-01 00:30:10|  2021-01-01 00:36:12|              1|          2.1|         1|                 N|         142|          43|           2|        8.0|  3.0|    0.5|       0.0|         0.0|                  0.3

## SECTION 4 : BASIC DATA EXPLORATION

In [3]:
print(f"Number of columns: {len(df_csv.columns)}")
print(f"Number of rows: {df_csv.count()}")

df_csv.select("VendorID", "fare_amount", "trip_distance").show(5)

Number of columns: 18
Number of rows: 1369765
+--------+-----------+-------------+
|VendorID|fare_amount|trip_distance|
+--------+-----------+-------------+
|       1|        8.0|          2.1|
|       1|        3.0|          0.2|
|       1|       42.0|         14.7|
|       1|       29.0|         10.6|
|       2|       16.5|         4.94|
+--------+-----------+-------------+
only showing top 5 rows



## SECTION 5 : SAVE AS PARQUET (UNCOMPRESSED)

In [4]:
import time

start_time = time.time()

df_csv.write.mode("overwrite") \
    .option("compression", "uncompressed") \
    .parquet("hdfs://namenode:9000/output/parquet_uncompressed")

end_time = time.time()
time_uncompressed = end_time - start_time
print(f"Write time for uncompressed Parquet: {time_uncompressed} seconds")

Write time for uncompressed Parquet: 17.965465545654297 seconds


## SECTION 6 : SAVE AS PARQUET (SNAPPY)

In [5]:
start_time = time.time()

df_csv.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("hdfs://namenode:9000/output/parquet_snappy")

end_time = time.time()
time_snappy = end_time - start_time
print(f"Write time for Snappy Parquet: {time_snappy} seconds")

Write time for Snappy Parquet: 11.231634140014648 seconds


## SECTION 7 : SAVE AS PARQUET (GZIP)

In [6]:
start_time = time.time()

df_csv.write.mode("overwrite") \
    .option("compression", "gzip") \
    .parquet("hdfs://namenode:9000/output/parquet_gzip")

end_time = time.time()
time_gzip = end_time - start_time
print(f"Write time for Gzip Parquet: {time_gzip} seconds")

Write time for Gzip Parquet: 13.93643307685852 seconds


## SECTION 8 : SIZE COMPARISON

The following commands must be executed manually in the terminal to check the sizes on HDFS:

```bash
hdfs dfs -du -h /data
hdfs dfs -du -h /output/parquet_uncompressed
hdfs dfs -du -h /output/parquet_snappy
hdfs dfs -du -h /output/parquet_gzip
```

| Format | Size | Compression Ratio |
|---|---|---|
| CSV | | |
| Parquet Uncompressed | | |
| Parquet Snappy | | |
| Parquet Gzip | | |

## SECTION 9 : BENCHMARK QUERY ON CSV

In [7]:
from pyspark.sql.functions import avg

start_time = time.time()

csv_result = df_csv.filter(df_csv["VendorID"] == 1).select(avg("fare_amount")).collect()

end_time = time.time()
csv_query_time = end_time - start_time

print(f"Result: {csv_result[0][0]}")
print(f"Query time on CSV: {csv_query_time} seconds")

Result: 11.145971657553122
Query time on CSV: 5.229201078414917 seconds


## SECTION 10 : BENCHMARK QUERY ON PARQUET

| Format | Execution Time |
|---|---|
| CSV | |
| Parquet (Snappy) | |

In [8]:
df_parquet = spark.read.parquet("hdfs://namenode:9000/output/parquet_snappy")

start_time = time.time()

parquet_result = df_parquet.filter(df_parquet["VendorID"] == 1).select(avg("fare_amount")).collect()

end_time = time.time()
parquet_query_time = end_time - start_time

print(f"Result: {parquet_result[0][0]}")
print(f"Query time on Parquet: {parquet_query_time} seconds")

Result: 11.145971657553122
Query time on Parquet: 1.751943826675415 seconds


## SECTION 11 : COLUMN PRUNING ANALYSIS

**Column Pruning:**
* Spark reads only the required column from the Parquet file.
* Unnecessary columns are skipped during the read operation.
* This significantly reduces I/O operations and memory usage.

In [9]:
parquet_df = spark.read.parquet("hdfs://namenode:9000/output/parquet_snappy")
parquet_df.select("fare_amount").explain(True)

== Parsed Logical Plan ==
'Project ['fare_amount]
+- Relation [VendorID#356,tpep_pickup_datetime#357,tpep_dropoff_datetime#358,passenger_count#359,trip_distance#360,RatecodeID#361,store_and_fwd_flag#362,PULocationID#363,DOLocationID#364,payment_type#365,fare_amount#366,extra#367,mta_tax#368,tip_amount#369,tolls_amount#370,improvement_surcharge#371,total_amount#372,congestion_surcharge#373] parquet

== Analyzed Logical Plan ==
fare_amount: double
Project [fare_amount#366]
+- Relation [VendorID#356,tpep_pickup_datetime#357,tpep_dropoff_datetime#358,passenger_count#359,trip_distance#360,RatecodeID#361,store_and_fwd_flag#362,PULocationID#363,DOLocationID#364,payment_type#365,fare_amount#366,extra#367,mta_tax#368,tip_amount#369,tolls_amount#370,improvement_surcharge#371,total_amount#372,congestion_surcharge#373] parquet

== Optimized Logical Plan ==
Project [fare_amount#366]
+- Relation [VendorID#356,tpep_pickup_datetime#357,tpep_dropoff_datetime#358,passenger_count#359,trip_distance#360,Ra

## SECTION 12 : DATA SKEW ANALYSIS

**Data Skew:** Data skew happens when data is unevenly distributed across partitions. Some nodes process huge amounts of data while others are idle.

You can check the distribution manually using this command in your terminal:
```bash
hdfs fsck /output/parquet_snappy -files -blocks -locations
```

* **Balanced distribution:** Blocks are roughly the same size and evenly spread across nodes, leading to optimal parallel processing.
* **Unbalanced distribution:** Some blocks are disproportionately large, causing straggler tasks that delay the entire job.
* **Possible bottlenecks:** High memory and CPU usage on overloaded nodes, causing network congestion and disk I/O wait times.

## SECTION 13 : RESULTS SUMMARY

### Storage Comparison
| Format | Storage Size |
|---|---|
| CSV | |
| Parquet Uncompressed | |
| Parquet Snappy | |
| Parquet Gzip | |

### Speed Comparison (Write)
| Format | Write Time |
|---|---|
| Parquet Uncompressed | |
| Parquet Snappy | |
| Parquet Gzip | |

### Compression Comparison
| Format | Compression Ratio |
|---|---|
| CSV | |
| Parquet Uncompressed | |
| Parquet Snappy | |
| Parquet Gzip | |

## SECTION 14 : CONCLUSION

* **CSV vs Parquet:** Parquet is columnar, which makes it much more efficient for analytical queries than CSV (row-based) because it allows for column pruning and better compression.
* **Snappy vs Gzip:** Snappy is optimized for speed and is the default for Spark. Gzip offers better compression ratios but is more CPU-intensive and slower.
* **Storage savings:** Parquet significantly reduces storage costs by leveraging columnar compression.
* **Performance improvement:** Parquet limits I/O by only reading necessary columns, significantly speeding up query times.
* **Importance for organizations such as SNIM and SOMELEC:** These organizations generate massive volumes of operational data. Using efficient storage formats like Parquet ensures faster reporting, reduced storage costs, and better scalability for their data lakes.
* **Strategic decision:** Choosing the right format impacts performance, infrastructure costs, and the overall agility of a Big Data platform.

--- 
## Commands Executed Outside Notebook

```bash
docker compose up -d
hdfs dfs -mkdir -p /data
hdfs dfs -put nyc_taxi.csv /data/
hdfs dfs -ls /data
hdfs dfs -du -h /output/parquet_uncompressed
hdfs dfs -du -h /output/parquet_snappy
hdfs dfs -du -h /output/parquet_gzip
hdfs fsck /output/parquet_snappy -files -blocks -locations
```